#### A/B Test Analysis

In [3]:
# Importing Libraries

import pandas as pd
import numpy as np
from scipy import stats

In [5]:
# Loading Data

df = pd.read_csv('/content/ab_stats.csv')
df.head()

,exposure_id,variant,customer_segment,clicked,purchased_7d,total_revenue_generated
0,EXP00000001,control,At Risk,0,0,0.0
1,EXP00000002,treatment,NaN,0,0,0.0
2,EXP00000003,treatment,NaN,0,0,0.0
3,EXP00000004,control,Promising,1,0,0.0
4,EXP00000005,treatment,Champions,0,0,0.0


In [6]:
# Handling NULL values

df['customer_segment'] = df['customer_segment'].fillna('Unknown')

##### Evaluvating Statistical Significance

In [8]:
# Calculating SUmmary Statistics per variant
summary = df.groupby('variant').agg(
    Exposures=('exposure_id', 'count'),
    CTR_pct=('clicked', lambda x: x.mean() * 100),
    CVR_7d_pct=('purchased_7d', lambda x: x.mean() * 100),
    Avg_Revenue=('total_revenue_generated', 'mean')
).round(4)

In [18]:
# Chi-Square Test for Categorical Data (Control/Treatment vs. Clicked/Not Clicked)

# Null Hypothesis: User action is independent of the variant
# Alternative Hypothesis: User action is dependent of the variant
alpha = 0.05

In [16]:
# CTR (Chi-Square)
contingency_ctr = pd.crosstab(df['variant'], df['clicked'])
chi2, p_value, dof, exp = stats.chi2_contingency(contingency_ctr)

if p_value < alpha:
  print(f"User Action is Dependent of the Variant. (p-value: {round(p_value, 3)})")
else:
  print(f"User Action is Independent. (p-value: {round(p_value, 3)})")

User Action is Dependent of the Variant. (p-value: 0.033)


In [17]:
# Converstion Rate (Chi-Square)
contingency_ctr = pd.crosstab(df['variant'], df['purchased_7d'])
chi2, p_value, dof, exp = stats.chi2_contingency(contingency_ctr)

if p_value < alpha:
  print(f"User Action is Dependent of the Variant. (p-value: {round(p_value, 3)})")
else:
  print(f"User Action is Independent. (p-value: {round(p_value, 3)})")

User Action is Independent. (p-value: 0.563)


In [23]:
# Welch's T-Test for comparing mean values of two groups.

# Null Hypothesis: Mean revenue in the Treatment group is equal to mean revenue in control group.
# Alternative Hypothesis: Mean revenue in the Treatment group is not equal to mean revenue in control group.
alpha = 0.05

In [21]:
# Average Revenue (Welch's T-Test)
control_rev = df[df['variant'] == 'control']['total_revenue_generated']
treatment_rev = df[df['variant'] == 'treatment']['total_revenue_generated']
t_stat, p_value = stats.ttest_ind(treatment_rev, control_rev, equal_var=False)

if p_value < alpha:
  print(f"Mean revenue of both group is same. (p-value: {round(p_value, 3)})")
else:
  print(f"Mean revenue of both group is different. (p-value: {round(p_value, 3)})")

Mean revenue of both group is different. (p-value: 0.567)


#### Segment Analysis

In [24]:
# Calculating Conversion Rate and Revenue Impact by Segment

segments = df['customer_segment'].unique()

for segment in segments:
    segment_data = df[df['customer_segment'] == segment]

    if len(segment_data) < 100:
        continue

    # Calculating Segment Baselines
    cvr_rates = segment_data.groupby('variant')['purchased_7d'].mean() * 100
    rev_rates = segment_data.groupby('variant')['total_revenue_generated'].mean()

    ctrl_cvr = cvr_rates.get('control', 0)
    trt_cvr = cvr_rates.get('treatment', 0)
    cvr_lift = trt_cvr - ctrl_cvr

    ctrl_rev = rev_rates.get('control', 0)
    trt_rev = rev_rates.get('treatment', 0)
    rev_lift = trt_rev - ctrl_rev

In [25]:
# Chi-Square Test for CVR

# Null Hypothesis: 7-day conversion rate is independent of the variant
# Alternative Hypothesis: 7-day Conversion rate is dependent of the variant
alpha = 0.05

In [40]:
# 7-day Converstion Rate (Chi-Square)
contingency = pd.crosstab(segment_data['variant'], segment_data['purchased_7d'])
try:
  chi2, p_val_cvr, dof, exp = stats.chi2_contingency(contingency)
  sig_cvr = "YES" if p_val_cvr < alpha else "NO"
except ValueError:
  p_val_cvr = 1.0
  sig_cvr = "Invalid/Not enough data"

In [41]:
# T-Test for comparing mean values of two groups.

# Null Hypothesis: Mean revenue in the Treatment group is equal to mean revenue in control group.
# Alternative Hypothesis: Mean revenue in the Treatment group is not equal to mean revenue in control group.
alpha = 0.05

In [42]:
# Average Revenue (Welch's T-Test)
control_rev = df[df['variant'] == 'control']['total_revenue_generated']
treatment_rev = df[df['variant'] == 'treatment']['total_revenue_generated']

try:
  t_stat, p_val_rev = stats.ttest_ind(treatment_rev, control_rev, equal_var=False)
  sig_rev = "YES" if p_val_rev < alpha else "NO"
except ValueError:
  p_value = 1.0
  sig_rev = "Invalid"


In [43]:
print(f"Segment: {segment.upper()} (n={len(segment_data)})")
print(f"  CVR Control: {ctrl_cvr:.2f}% | Treatment: {trt_cvr:.2f}% | Lift: {cvr_lift:+.2f}% | Significant? {sig_cvr} (p={p_val_cvr:.4f})")
print(f"  Rev Control: ${ctrl_rev:.2f} | Treatment: ${trt_rev:.2f} | Lift: ${rev_lift:+.2f} | Significant? {sig_rev} (p={p_val_rev:.4f})")

Segment: NEW CUSTOMERS (n=277)
  CVR Control: 0.00% | Treatment: 0.00% | Lift: +0.00% | Significant? NO (p=1.0000)
  Rev Control: $0.00 | Treatment: $0.00 | Lift: $+0.00 | Significant? NO (p=0.5669)
